# Tier 1 — grading every hierarchy metric against a tree we built ourselves

A metric that keeps a lot of edges has proved nothing. It could be measuring the
hierarchy, or it could be measuring which features happen to fire often. On
`gemma-2-2b` there is no way to tell the two apart: the true parent→child tree is
exactly what the project is trying to find out, so any answer the battery gives is
also the only thing there is to check it against.

So the answer is fixed first. `validation/synthetic_toy_world.py` declares a
5-parent tree over 42 features and then **injects six structures on purpose** —
three pathologies each metric is supposed to reject, and three that expose what
the battery cannot see. It reduces that world to the *same cached statistics*
`run_metrics.analyse_pair` reads off `exp0_stats.pt`, so the production metric
functions run unchanged, at the production thresholds in `config.py`.
`validation/calibrate_on_synthetic_toy.py` then grades each one on the job it
claims, and every row is a **pass/fail plus a margin** — how decisively it
separated the two classes.

This notebook is a reading of those two files, not a fork of them. Every number
below comes from calling their functions; nothing is reimplemented here, because a
notebook that recomputes a metric its own way can agree with itself while
disagreeing with what ships.

| | question | file it reads |
| --- | --- | --- |
| §1–2 | what is in the world, and what does it reduce to? | `synthetic_toy_world.py` |
| §3–4 | which edges does coverage propose, and what prunes them? | `calibrate_on_synthetic_toy.py` |
| §5–6 | did each metric do its own job? | the scorecard |
| §7 | what does the battery provably *not* catch? | the two negative controls |
| §8 | does any of it depend on the seed? | neither file — the sweep is new here |

**Where this sits.** Tier 1 is certain and artificial: it proves the arithmetic is
right and says nothing about whether an SAE would ever learn such a structure.
Tier 2 closes exactly that gap by putting a *trained* SAE in front of the same
battery — [`../calibrate_on_trained_toy.py`](../calibrate_on_trained_toy.py) for the
script, and the training notebook beside this one for the run that produces the
checkpoint. The two tiers do not share a world — see
[`validation/README.md`](../README.md).

## Setup

Pure CPU, a few seconds end to end — the world is 2,395 tokens and 42 features,
and no model is loaded. Needs `torch`, `plotly` and `numpy`; the figure export in
§9 additionally needs `kaleido`.

Run from anywhere: the next cell walks up to the repo root (the directory holding
`config.py`) and works from there, which is what makes `import config` and
`from validation... import` resolve. That is also why this notebook survived being
moved into `validation/notebooks/` — nothing below counts `.parent` hops.

In [78]:
import os
import sys
from pathlib import Path

# Anchor on the metrics repo root rather than counting `.parent` hops, so moving
# this notebook does not break the imports. Idempotent: safe to re-run.
_start = Path.cwd().resolve()
# Walk up first, then look for a metrics/ child on the way up: a kernel launched
# from the SOAR workspace root (VS Code's default when the folder opened is the
# workspace, not the repo) has no config.py on any ancestor, and the bare
# `next(...)` raised StopIteration there instead of finding the repo one level down.
ROOT = next((p for p in [_start, *_start.parents] if (p / "config.py").is_file()),
            None)
if ROOT is None:
    ROOT = next((p / "metrics" for p in [_start, *_start.parents]
                 if (p / "metrics" / "config.py").is_file()), None)
if ROOT is None:
    raise SystemExit(f"cannot find the metrics repo (no config.py at or above {_start})")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import torch

import config as C

print("repo root :", ROOT)
print("torch     :", torch.__version__)
print()
print("production thresholds this calibration runs at (config.py):")
for name in ["FIRE_THRESHOLD", "EDGE_TAU", "MIN_FIRE_COUNT", "MIN_JOINT",
             "RECON_REL_GAIN_MIN", "FREQ_SURVIVAL_MIN", "FREQ_HIGH_MASS",
             "FREQ_MID_MASS", "SIBLING_REDUNDANCY_FLAG", "SUPERPARENT_OUTDEG_FRAC",
             "SUPERPARENT_FIRE_FRAC", "SRES_RANK_TOP_K"]:
    print(f"  {name:<26} = {getattr(C, name)}")

repo root : /Users/ruqiya/Codeing-repos/Research/eleuther/soar-eleuther-i6-hierarchy/metrics
torch     : 2.13.0

production thresholds this calibration runs at (config.py):
  FIRE_THRESHOLD             = 0.001
  EDGE_TAU                   = 0.5
  MIN_FIRE_COUNT             = 20
  MIN_JOINT                  = 30
  RECON_REL_GAIN_MIN         = 0.01
  FREQ_SURVIVAL_MIN          = 0.5
  FREQ_HIGH_MASS             = 0.5
  FREQ_MID_MASS              = 0.4
  SIBLING_REDUNDANCY_FLAG    = 0.5
  SUPERPARENT_OUTDEG_FRAC    = 0.3
  SUPERPARENT_FIRE_FRAC      = 0.1
  SRES_RANK_TOP_K            = 5


## 1. The world, by declaration

Ten parent-block features (`P = 10`) and thirty-two child-block features
(`C = 32`). The block split is not decoration: the production pipeline scores one
**block pair** at a time, because Matryoshka's nesting is what gives an edge its
direction, so the toy is shaped like the smallest thing `analyse_pair` accepts.

Five parents own four children each — twenty genuine edges, and the only edges
that exist. Everything else in the declaration is there to be caught, or to prove
it cannot be:

| # | structure | what it looks like | which metric should react |
| --- | --- | --- | --- |
| A | **superparent** (parent 7) | fires on ~90% of *all* tokens with a tiny activation | out-degree flags it; reconstruction rejects its edges |
| B | **frequency coincidence** (6 → 20) | the pair only co-fires on the single most frequent token id | only the token-frequency control |
| C | **feature split** (5 → 24, 25, 26) | three near-duplicate children, same tokens, same activation | sibling redundancy and energy concentration |
| D | **absorption** (8 → 21) | a real refinement where the parent is *silent* on the child's tokens | *nothing* — negative control |
| E | **shared topic** (9 → 22) | two features driven by a third thing, neither refining the other | *nothing* — negative control |
| F | **within-block** (27 → 28, and 29 ≡ 30) | containment and co-extension inside one block, where no block order helps | in-block directed coverage |

D and E are the two open columns in the properties matrix. They are scored as
rows that **pass when the battery does nothing**, which turns "we know this is a
blind spot" from an assertion into something a regression would break.

In [79]:
from validation.synthetic_toy_world import (
    ABSORB_CHILD, ABSORB_PARENT, C as N_CHILD, D_MODEL, FREQ_CHILD, FREQ_PARENT,
    GENUINE_TREE, IN_BLOCK_CHILD, IN_BLOCK_DUP, IN_BLOCK_PARENT, P as N_PARENT,
    SPLIT_CHILDREN, SPLIT_PARENT, SUPERPARENT, TOPIC_CHILD, TOPIC_PARENT,
    build_world,
)

print(f"{N_PARENT} parent-block features, {N_CHILD} child-block features, "
      f"d_model = {D_MODEL}\n")
print("genuine tree (parent-local -> child-locals):")
for p, kids in GENUINE_TREE.items():
    print(f"  {p} -> {kids}")
print(f"\n  {len(GENUINE_TREE) * 4} genuine edges, and no others\n")
print("injected on purpose:")
print(f"  A superparent          : parent {SUPERPARENT}")
print(f"  B frequency coincidence: {FREQ_PARENT} -> {FREQ_CHILD}")
print(f"  C feature split        : {SPLIT_PARENT} -> {SPLIT_CHILDREN}")
print(f"  D absorption           : {ABSORB_PARENT} -> {ABSORB_CHILD}   (negative control)")
print(f"  E shared topic         : {TOPIC_PARENT} -> {TOPIC_CHILD}   (negative control)")
print(f"  F within-block         : {IN_BLOCK_PARENT} -> {IN_BLOCK_CHILD}, "
      f"duplicate pair {IN_BLOCK_DUP}")

10 parent-block features, 32 child-block features, d_model = 64

genuine tree (parent-local -> child-locals):
  0 -> [0, 1, 2, 3]
  1 -> [4, 5, 6, 7]
  2 -> [8, 9, 10, 11]
  3 -> [12, 13, 14, 15]
  4 -> [16, 17, 18, 19]

  20 genuine edges, and no others

injected on purpose:
  A superparent          : parent 7
  B frequency coincidence: 6 -> 20
  C feature split        : 5 -> [24, 25, 26]
  D absorption           : 8 -> 21   (negative control)
  E shared topic         : 9 -> 22   (negative control)
  F within-block         : 27 -> 28, duplicate pair (29, 30)


### The same thing as a picture

Parent block on the left, child block on the right, one row per child. The links
are ground truth, coloured by which of the six structures they belong to — no
metric has run yet.

The superparent's co-firings are drawn faintly because they are **not edges**:
they are the thirty pairs its 90% firing rate will hand to coverage in §3, and
seeing them as a wash across the whole column is the point. The two links inside
the child block bow off to the right, since a straight line between two nodes in
the same column would run through everything between them.

In [80]:
import plotly.graph_objects as go

# Same palette as the Tier-2 training notebook in this directory, so a reader moving
# between the two is not relearning the colours.
BLUE, GREY, ORANGE = "#3b6ea5", "#9aa0a6", "#e8833a"
GREEN, RED, PURPLE = "#2e8b57", "#c0392b", "#7d3c98"
AMBER, INK = "#c9820f", "#2B2B33"

# role -> (colour, short name), for both the nodes and the links
ROLE = {
    "genuine":     (GREEN,   "genuine tree"),
    "split":       (PURPLE,  "feature split (C)"),
    "freq":        (AMBER,   "frequency coincidence (B)"),
    "absorbed":    (RED,     "absorption (D)"),
    "topic":       (BLUE,    "shared topic (E)"),
    "in_block":    (ORANGE,  "within-block (F)"),
    # A dedicated grey, darker than the GREY used for node outlines: a 30-line wash
    # has to be readable as grey, and at hairline width #9aa0a6 renders as white.
    "superparent": ("#767c83", "superparent (A)"),
    "unused":      ("#d9dde1", "declared but never fires"),
}

GENUINE_EDGES = sorted((p, c) for p, kids in GENUINE_TREE.items() for c in kids)


def _arc(p0, p1, bow=0.45, n_points=40):
    """Quadratic Bezier between two points, bowed perpendicular to the straight line.

    Borrowed from the Tier-2 notebook for the same reason it exists there: a link
    between two nodes in the same column would otherwise be drawn straight through
    every node in between and read as a chain of edges that does not exist.
    """
    (x0, y0), (x1, y1) = p0, p1
    dx, dy = x1 - x0, y1 - y0
    length = (dx * dx + dy * dy) ** 0.5 or 1.0
    cx = (x0 + x1) / 2 - dy / length * bow * length
    cy = (y0 + y1) / 2 + dx / length * bow * length
    ts = [i / (n_points - 1) for i in range(n_points)]
    return ([(1 - t) ** 2 * x0 + 2 * (1 - t) * t * cx + t * t * x1 for t in ts],
            [(1 - t) ** 2 * y0 + 2 * (1 - t) * t * cy + t * t * y1 for t in ts])


# Parents are spread over the child column's span so the two blocks read as one
# figure rather than a tall column beside a short one.
PY = {p: -(p + 0.5) * N_CHILD / N_PARENT for p in range(N_PARENT)}
CY = {c: -(c + 0.5) for c in range(N_CHILD)}
PX, CX = 0.0, 1.0


def parent_role(p):
    if p in GENUINE_TREE:
        return "genuine"
    return {SPLIT_PARENT: "split", FREQ_PARENT: "freq", SUPERPARENT: "superparent",
            ABSORB_PARENT: "absorbed", TOPIC_PARENT: "topic"}[p]


def child_role(c):
    if any(c in kids for kids in GENUINE_TREE.values()):
        return "genuine"
    if c in SPLIT_CHILDREN:
        return "split"
    if c in (IN_BLOCK_PARENT, IN_BLOCK_CHILD, *IN_BLOCK_DUP):
        return "in_block"
    # Slots 23 and 31: capacity the declaration never fills. Colouring them like
    # the within-block group would invent a structure the world does not have.
    return {FREQ_CHILD: "freq", ABSORB_CHILD: "absorbed", TOPIC_CHILD: "topic"}.get(
        c, "unused")


def draw_world(show_superparent=True):
    fig = go.Figure()
    seen = set()

    def link(p, c, role, dash="solid", width=2.0, opacity=1.0):
        colour, name = ROLE[role]
        fig.add_trace(go.Scatter(
            x=[PX, CX], y=[PY[p], CY[c]], mode="lines",
            line=dict(color=colour, width=width, dash=dash), opacity=opacity,
            legendgroup=role, showlegend=False,
            hovertext=f"{p} -> {c}  ({name}"
                      + (", co-firing only — not an edge)" if role == "superparent"
                         else ")"), hoverinfo="text"))
        seen.add(role)

    if show_superparent:                      # drawn first, so it sits behind
        for c in range(N_CHILD):
            link(SUPERPARENT, c, "superparent", width=1.0, opacity=0.55)
    for p, c in GENUINE_EDGES:
        link(p, c, "genuine", width=2.0)
    for c in SPLIT_CHILDREN:
        link(SPLIT_PARENT, c, "split", width=2.0)
    link(FREQ_PARENT, FREQ_CHILD, "freq", width=2.0)
    link(ABSORB_PARENT, ABSORB_CHILD, "absorbed", dash="dash", width=2.0)
    link(TOPIC_PARENT, TOPIC_CHILD, "topic", dash="dot", width=2.0)

    # (F) lives entirely inside the child column, so both links bow outward.
    for (a, b), dash, tag in (((IN_BLOCK_PARENT, IN_BLOCK_CHILD), "solid", "contains"),
                              (IN_BLOCK_DUP, "dot", "co-extensive")):
        xs, ys = _arc((CX, CY[a]), (CX, CY[b]))
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode="lines", line=dict(color=ORANGE, width=2, dash=dash),
            legendgroup="in_block", showlegend=False,
            hovertext=f"{a} — {b}  (within-block, {tag})", hoverinfo="text"))
        seen.add("in_block")

    for xs, ys, roles, ids, side in (
            ([PX] * N_PARENT, [PY[p] for p in range(N_PARENT)],
             [parent_role(p) for p in range(N_PARENT)], list(range(N_PARENT)), "parent"),
            ([CX] * N_CHILD, [CY[c] for c in range(N_CHILD)],
             [child_role(c) for c in range(N_CHILD)], list(range(N_CHILD)), "child")):
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode="markers+text", showlegend=False,
            text=[str(i) for i in ids], textposition="middle center",
            textfont=dict(color="white", size=9),
            marker=dict(size=22, color=[ROLE[r][0] for r in roles],
                        line=dict(width=1, color="white")),
            hovertext=[f"{side}-local {i} — {ROLE[r][1]}" for i, r in zip(ids, roles)],
            hoverinfo="text"))

    # --- the colour key, built as three named groups -------------------------
    # Drawn as proxies (x=[None]) rather than left to the first real trace of each
    # colour, because the order a legend comes out in is then the order the edges
    # happened to be drawn in, and the grouping -- healthy / injected / uncatchable
    # -- is the reading this figure is for. Square markers carry the colour, the
    # line behind them carries the dash, since both mean something here.
    # Four groups, not three. "nothing catches these" is true of the two negative
    # controls and false of the within-block pair: no parent->child edge can express
    # structure inside one block, but `in_block_edges.directed_coverage` scores it and
    # passes (metric 7 in the scorecard). Filing (F) under the blind spots reported a
    # working metric as a gap in the battery.
    KEY = [
        ("the genuine tree", [("genuine", "solid")]),
        ("injected pathologies — a metric catches each",
         [("superparent", "solid"), ("freq", "solid"), ("split", "solid")]),
        ("blind spots — nothing catches these",
         [("absorbed", "dash"), ("topic", "dot")]),
        ("scored by a different metric", [("in_block", "solid")]),
    ]
    for group, entries in KEY:
        for role, dash in entries:
            colour, label = ROLE[role]
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode="lines+markers", name=label,
                legendgroup=group, legendgrouptitle=dict(text=f"<b>{group}</b>"),
                line=dict(color=colour, width=2.2, dash=dash),
                marker=dict(symbol="square", size=11, color=colour),
                hoverinfo="skip"))

    fig.update_layout(
        title="The toy world as declared — 20 genuine edges and six injected structures",
        width=860, height=int(120 + 24 * N_CHILD) + 110, plot_bgcolor="white",
        xaxis=dict(visible=False, range=[-0.35, 1.55]),
        yaxis=dict(visible=False, range=[-N_CHILD - 0.6, 0.6]),
        margin=dict(l=20, r=20, t=60, b=190),
        legend=dict(orientation="h", yanchor="top", y=-0.02, x=0,
                    font=dict(size=11), tracegroupgap=14,
                    grouptitlefont=dict(size=11.5, color=INK)))
    fig.add_annotation(x=PX, y=0.4, text="<b>parent block</b>", showarrow=False,
                       font=dict(size=12, color=INK))
    fig.add_annotation(x=CX, y=0.4, text="<b>child block</b>", showarrow=False,
                       font=dict(size=12, color=INK))
    return fig


world_fig = draw_world()
world_fig.show()

## 2. From a token list to cached statistics

`build_world` writes a corpus one token at a time — each token is a dict of
`feature → activation` plus a token id — and then reduces it with `_reduce` into
the tensors the metrics actually read. The reduction is the part worth checking,
because it is what makes this toy interchangeable with a real run: `cofire`,
`g_parent_sum`, `err_sum_c`, `cofire_by_bucket` and the schema-v2 energy
accumulators are the same fields, in the same shapes, that
`collect_statistics.py` writes to `exp0_stats.pt` for gemma.

Two modelling choices carry weight:

- **The residual error is isotropic gaussian noise**, drawn independently of the
  feature directions. The per-token ablation gain
  `g_f = 2·a_f·⟨d_f, err⟩ + a_f²·‖d_f‖²` then reduces in expectation to `a_f²`,
  so reconstruction gain is driven by *activation magnitude*. That is exactly the
  knob separating a real refinement from a superparent riding on frequency, so
  metric 2 is exercised on the property it claims to test.
- **`D_MODEL = 64`** is set by the probe rank rule and nothing else. The rule asks
  whether the parent's decoder direction is in the top-*k* correlations over all
  42 features, which only means "parenthood" if unrelated directions are roughly
  orthogonal. Random unit vectors in *d* dimensions correlate by about
  `sqrt(2/(pi·d))` — 0.20 at *d* = 16, where measured runs had genuine edges
  failing with the parent pushed to rank 6–8 by features it has nothing to do
  with. That is a fact about 42 directions in 16 dimensions, not about the metric.

`build_world` also returns a **per-token view** — `resid`, `fired`, `W_dec` —
which the four probe/redundancy functions read instead of the reduced counts.
Until 7 August those four had no ground-truth calibration anywhere; the claim that
Tier 2 covered them was false, since Tier 2 imports coverage, reconstruction and
the frequency control and nothing else.

In [81]:
stats, labels = build_world(seed=0)

print(f"corpus: {stats['total_tokens']} tokens over "
      f"{int(stats['token_counts'].shape[0])} distinct token ids\n")

reduced = ["fire_count", "cofire", "g_parent_sum", "err_sum_c", "g_child_sum",
           "within_cofire", "energy_cofire", "energy_total", "union_count",
           "union_energy", "cofire_by_bucket", "fire_c_by_bucket"]
per_token = ["resid", "fired", "W_dec", "tok_ids"]

print("reduced statistics — the same fields collect_statistics.py caches for gemma:")
for k in reduced:
    print(f"  {k:<20} {tuple(stats[k].shape)}")
print("\nper-token view — what the probe and redundancy functions read instead:")
for k in per_token:
    print(f"  {k:<20} {tuple(stats[k].shape)}")

print("\nground-truth labels carried alongside:")
for name in ["genuine", "freq_edges", "split_children", "superparent_parents",
             "split_parents", "absorbed_edges", "topical_edges", "in_block_edges",
             "in_block_duplicates"]:
    print(f"  {name:<20} {sorted(getattr(labels, name))}")

corpus: 2395 tokens over 683 distinct token ids

reduced statistics — the same fields collect_statistics.py caches for gemma:
  fire_count           (42,)
  cofire               (10, 32)
  g_parent_sum         (10, 32)
  err_sum_c            (32,)
  g_child_sum          (32,)
  within_cofire        (32, 32)
  energy_cofire        (10, 32)
  energy_total         (10,)
  union_count          (10,)
  union_energy         (10,)
  cofire_by_bucket     (3, 10, 32)
  fire_c_by_bucket     (3, 32)

per-token view — what the probe and redundancy functions read instead:
  resid                (2395, 64)
  fired                (2395, 42)
  W_dec                (42, 64)
  tok_ids              (2395,)

ground-truth labels carried alongside:
  genuine              [(0, 0), (0, 1), (0, 2), (0, 3), (1, 4), (1, 5), (1, 6), (1, 7), (2, 8), (2, 9), (2, 10), (2, 11), (3, 12), (3, 13), (3, 14), (3, 15), (4, 16), (4, 17), (4, 18), (4, 19)]
  freq_edges           [(6, 20)]
  split_children       [24, 25, 26]


### Who fires, and how often

The superparent is visible without any metric at all: it fires on 91% of tokens,
while every genuine child fires on 40. That gap is the whole of pathology (A), and
it is why the y-axis below is logarithmic.

The right panel is the token-frequency design that pathology (B) rides on.
`frequency_buckets` walks token ids from most to least frequent and cuts at
cumulative mass 0.50 and 0.90, so the **high bucket here holds a single token id**
— id 0, on its own, is 39% of the corpus. The frequency-coincidence pair co-fires
only there.

In [82]:
from plotly.subplots import make_subplots

fire = stats["fire_count"]
n_tok = stats["total_tokens"]
roles = ([parent_role(p) for p in range(N_PARENT)]
         + [child_role(c) for c in range(N_CHILD)])
names = ([f"parent {p}" for p in range(N_PARENT)]
         + [f"child {c}" for c in range(N_CHILD)])

buckets, counts = stats["buckets"], stats["token_counts"]
bucket_ids = [int((buckets == k).sum()) for k in range(3)]
bucket_tok = [float(counts[buckets == k].sum()) for k in range(3)]

fig = make_subplots(rows=1, cols=2, column_widths=[0.68, 0.32],
                    horizontal_spacing=0.09,
                    subplot_titles=("tokens each feature fires on (log)",
                                    "corpus mass per frequency bucket"))

for role, (colour, label) in ROLE.items():
    idx = [i for i, r in enumerate(roles) if r == role]
    if not idx:
        continue
    fig.add_trace(go.Bar(
        x=[names[i] for i in idx], y=[float(fire[i]) for i in idx],
        marker_color=colour, name=label, legendgroup=role,
        hovertemplate="%{x}<br>%{y:.0f} tokens<extra></extra>"), row=1, col=1)

fig.add_trace(go.Bar(
    x=["high (bucket 0)", "mid (bucket 1)", "rare (bucket 2)"], y=bucket_tok,
    marker_color=[AMBER, BLUE, GREY], showlegend=False,
    text=[f"{n:.0f} tokens<br>{i} id{'s' if i != 1 else ''}<br>"
          f"{100 * n / n_tok:.0f}% of corpus"
          for n, i in zip(bucket_tok, bucket_ids)],
    textposition="outside",
    hovertemplate="%{x}<br>%{y:.0f} tokens<extra></extra>"), row=1, col=2)

fig.add_hline(y=C.MIN_FIRE_COUNT, line=dict(color=RED, width=1, dash="dash"),
              annotation_text=f"MIN_FIRE_COUNT = {C.MIN_FIRE_COUNT}",
              annotation_position="top left", row=1, col=1)
# An explicit range, because bars on a log axis are drawn from zero and plotly
# then autoscales the top to something absurd (1e20) to fit a bar with no bottom.
fig.update_yaxes(type="log", range=[0.9, 3.6], dtick=1, title_text="tokens",
                 row=1, col=1)
fig.update_yaxes(range=[0, max(bucket_tok) * 1.45], row=1, col=2)
# Traces are grouped by role, so pin the category order or the x axis comes out
# sorted by colour rather than by feature.
fig.update_xaxes(categoryorder="array", categoryarray=names,
                 tickangle=-60, tickfont=dict(size=8), row=1, col=1)
fig.update_layout(
    width=1150, height=460, plot_bgcolor="white", bargap=0.15,
    title=f"The corpus: {n_tok} tokens, 42 features",
    margin=dict(l=50, r=20, t=90, b=110),
    legend=dict(orientation="h", yanchor="bottom", y=-0.52, x=0, font=dict(size=10)))
corpus_fig = fig
corpus_fig.show()

print(f"superparent fires on {100 * float(fire[SUPERPARENT]) / n_tok:.1f}% of tokens; "
      f"a genuine child on {100 * float(fire[N_PARENT]) / n_tok:.1f}%")
silent = [i - N_PARENT for i in range(N_PARENT, len(fire)) if float(fire[i]) == 0]
print(f"child-block slots the world never uses: {silent} "
      f"(declared capacity the declaration does not fill)")

superparent fires on 91.3% of tokens; a genuine child on 1.7%
child-block slots the world never uses: [23, 31] (declared capacity the declaration does not fill)


## 3. Coverage decides what every other metric ever sees

Reverse coverage is `R[p, c] = cofire(p, c) / fire(c)` — of the tokens where the
child fires, what fraction also has the parent firing. Containment makes this 1 for
a genuine edge and small for an unrelated pair, which is why it, and not the
forward direction, is the candidate generator.

`keep_edges` then keeps a pair when `R ≥ EDGE_TAU` and both endpoints clear
`MIN_FIRE_COUNT`. That is a **gate, not a score**: a pair it drops is never handed
to reconstruction, the frequency control, or anything else. The absorption negative
control (D) is precisely a true edge that dies here, and no downstream metric gets
a chance to be blamed for it.

`_run_metrics` below is the calibration script's own function — it mirrors
`run_metrics.analyse_pair`'s calls on the toy's single block pair, so what runs
here is what runs on gemma.

In [83]:
from validation.calibrate_on_synthetic_toy import _run_metrics, _score, calibrate

m = _run_metrics(stats)

R, edge_mask = m["R"], m["edge_mask"]
kept = {(int(p), int(c)) for p, c in torch.nonzero(edge_mask).tolist()}
genuine = labels.genuine

print(f"candidate edges after coverage: {len(kept)}")
print(f"  genuine                     : {len(kept & genuine)}/{len(genuine)}")
print(f"  from the superparent        : {sum(p == SUPERPARENT for p, _ in kept)}")
print(f"  feature-split children      : {sum(p == SPLIT_PARENT for p, _ in kept)}")
print(f"  frequency coincidence       : {int((FREQ_PARENT, FREQ_CHILD) in kept)}")
print(f"  shared topic                : {int((TOPIC_PARENT, TOPIC_CHILD) in kept)}")
print(f"  absorbed true edge          : {int((ABSORB_PARENT, ABSORB_CHILD) in kept)} "
      f"  <- R = {float(R[ABSORB_PARENT, ABSORB_CHILD]):.2f}, below tau = {C.EDGE_TAU}")

candidate edges after coverage: 55
  genuine                     : 20/20
  from the superparent        : 30
  feature-split children      : 3
  frequency coincidence       : 1
  shared topic                : 1
  absorbed true edge          : 0   <- R = 0.00, below tau = 0.5


Every genuine edge is proposed, and so are thirty-five pairs that are not genuine.
That is the intended shape: coverage is meant to be permissive, and metrics 2–9
exist to prune what it lets through.

In the matrix below, a bright cell is a high reverse coverage. The green rings mark
the twenty genuine edges; a red cross marks a pair coverage kept that is not one.
Row 7 — the superparent — is bright almost the whole way across, and row 8's true
edge to child 21 is dark, which is what absorption looks like when only co-firing
is in view.

In [84]:
fig = go.Figure(go.Heatmap(
    z=R.numpy(), colorscale="Blues", zmin=0, zmax=1,
    colorbar=dict(title="R = cofire / fire(child)", thickness=14),
    hovertemplate="parent %{y} -> child %{x}<br>R = %{z:.2f}<extra></extra>"))

gx = [c for _, c in sorted(genuine)]
gy = [p for p, _ in sorted(genuine)]
fig.add_trace(go.Scatter(
    x=gx, y=gy, mode="markers", name="genuine edge (ground truth)",
    marker=dict(symbol="circle-open", size=13, color=GREEN, line=dict(width=2.2)),
    hoverinfo="skip"))

fx = [c for p, c in sorted(kept - genuine)]
fy = [p for p, c in sorted(kept - genuine)]
fig.add_trace(go.Scatter(
    x=fx, y=fy, mode="markers", name="kept by coverage, not genuine",
    marker=dict(symbol="x-thin", size=9, color=RED, line=dict(width=1.8, color=RED)),
    hoverinfo="skip"))

fig.add_trace(go.Scatter(
    x=[ABSORB_CHILD], y=[ABSORB_PARENT], mode="markers",
    name="true edge coverage cannot propose (D)",
    marker=dict(symbol="square-open", size=15, color=AMBER, line=dict(width=2.2)),
    hoverinfo="skip"))

fig.update_layout(
    title=f"Reverse coverage, and the {len(kept)} pairs it proposes "
          f"(tau = {C.EDGE_TAU})",
    width=1120, height=430, plot_bgcolor="white",
    xaxis=dict(title="child-block local index", dtick=1, tickfont=dict(size=9),
               range=[-0.5, N_CHILD - 0.5]),
    yaxis=dict(title="parent-block local index", dtick=1, autorange="reversed"),
    margin=dict(l=60, r=20, t=60, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=-0.42, x=0))
coverage_fig = fig
coverage_fig.show()

## 4. What survives each gate

Tier 1 grades each metric **separately**, on its own pathology — that is the
scorecard in §5, and it is the right way to answer "does this metric work?".
But it leaves a second question unanswered: what would be left if the gates were
composed the way a run actually composes them?

The composition below is the one Tier 2 uses — coverage, then the reconstruction
condition, then frequency survival. `run_metrics.py` itself reports the gates
side by side rather than intersecting them, so this is a derived view, not a
number the pipeline prints.

In [85]:
recon_ok = m["recon"]["passes"] & edge_mask
survival = m["fcov"]["survival"]
freq_ok = recon_ok & (survival >= C.FREQ_SURVIVAL_MIN)

CLASSES = [
    ("genuine",           GREEN,  lambda e: e in genuine),
    ("superparent (A)",   GREY,   lambda e: e[0] == SUPERPARENT),
    ("feature split (C)", PURPLE, lambda e: e[0] == SPLIT_PARENT),
    ("freq. coincidence (B)", AMBER, lambda e: e == (FREQ_PARENT, FREQ_CHILD)),
    ("shared topic (E)",  BLUE,   lambda e: e == (TOPIC_PARENT, TOPIC_CHILD)),
]
STAGES = [("after coverage", edge_mask),
          ("+ reconstruction", recon_ok),
          ("+ frequency control", freq_ok)]

fig = go.Figure()
for label, colour, belongs in CLASSES:
    xs = []
    for _, mask in STAGES:
        s = {(int(p), int(c)) for p, c in torch.nonzero(mask).tolist()}
        xs.append(sum(belongs(e) for e in s))
    fig.add_trace(go.Bar(
        y=[name for name, _ in STAGES], x=xs, orientation="h", name=label,
        marker_color=colour, text=[str(v) if v else "" for v in xs],
        textposition="inside", insidetextanchor="middle",
        textfont=dict(color="white", size=11),
        hovertemplate=f"{label}<br>%{{x}} edges<extra></extra>"))

fig.update_layout(
    barmode="stack", title="Composing the gates: 55 candidates in, 24 out — "
                           "and all 20 genuine edges survive",
    width=1050, height=330, plot_bgcolor="white",
    xaxis=dict(title="edges"), yaxis=dict(autorange="reversed"),
    margin=dict(l=150, r=20, t=60, b=70),
    legend=dict(orientation="h", yanchor="bottom", y=-0.35, x=0))
funnel_fig = fig
funnel_fig.show()

for name, mask in STAGES:
    s = {(int(p), int(c)) for p, c in torch.nonzero(mask).tolist()}
    print(f"{name:<22} {len(s):>3} edges | genuine {len(s & genuine):>2}/20")

after coverage          55 edges | genuine 20/20
+ reconstruction        25 edges | genuine 20/20
+ frequency control     24 edges | genuine 20/20


Reconstruction removes all thirty superparent pairs and keeps every genuine edge;
the frequency control then removes the one coincidence edge. What is left over the
twenty genuine edges is the feature split — three edges that are *real*
refinements, so no gate should drop them — and the shared-topic pair, which
nothing here can drop. Both are handled by metrics that **flag** rather than
filter, which is the next section.

### The same edge set, drawn as the tree

The funnel counts edges; this says *which* ones, against the tree the cached
statistics were built from — the same before/after pair as the paper's Tier-2
figure (`outputs/paper_figuers/calibration_toy_tree_recovered.png`), in the same
colours, so the two tiers read as one statement rather than two conventions.

**What the left panel draws is the tree, not the label set.** Those are not the
same thing, and the difference is deliberate in `synthetic_toy_world.py`:

- the twenty `labels.genuine` edges, plus
- **5 → 24, 25, 26**, which are *real* refinements — the pathology is that the
  three children are duplicates of each other, not that the parent is wrong, and
- **8 → 21**, which the module's own docstring calls "a real refinement", kept out
  of `labels.genuine` so that metrics 2–9 are not scored on a candidate coverage
  never hands them.

Twenty-four true edges, then. `labels.genuine` is the subset the scorecard grades,
which is why §5 says twenty.

In [86]:
# --- the ground-truth tree: every real parent -> child edge in the world -------
TREE_CHILDREN = {**{p: list(kids) for p, kids in GENUINE_TREE.items()},
                 SPLIT_PARENT: list(SPLIT_CHILDREN),      # real edges, duplicate children
                 ABSORB_PARENT: [ABSORB_CHILD]}           # real edge, absorbed child
# (6 -> 20) and (9 -> 22) are deliberately absent: a frequency coincidence and a
# shared topic are not edges, which is the whole reason they were injected.
TREE_EDGES = sorted((p, c) for p, kids in TREE_CHILDREN.items() for c in kids)
ORPHANS = [c for c in range(N_CHILD)
           if not any(c in kids for kids in TREE_CHILDREN.values())]

survivors = {(int(p), int(c)) for p, c in torch.nonzero(freq_ok).tolist()}
tp = survivors & set(TREE_EDGES)
fn = set(TREE_EDGES) - survivors
fp = survivors - set(TREE_EDGES)


def forest_layout():
    """Left-to-right: depth on x, one slot per row on y.

    A parent takes the midpoint of the span its children occupy; a parent with no
    children, and every child with no parent, takes a row of its own. Same shape as
    the paper figure's layout, so the two are comparable panel for panel.
    """
    slot, ppos, cpos = 0.0, {}, {}
    for parent in range(N_PARENT):
        kids = TREE_CHILDREN.get(parent, [])
        if not kids:
            ppos[parent] = slot
            slot += 1
            continue
        for c in kids:
            cpos[c] = slot
            slot += 1
        ppos[parent] = (min(cpos[c] for c in kids) + max(cpos[c] for c in kids)) / 2
    slot += 0.8                                   # a gap before the parentless rows
    for c in ORPHANS:
        cpos[c] = slot
        slot += 1
    return ppos, cpos, slot


PPOS, CPOS, N_ROWS = forest_layout()
PT = {p: (0.0, -y) for p, y in PPOS.items()}
CT = {c: (1.0, -y) for c, y in CPOS.items()}

# Outcome colours from reporting/make_report_figures.py, so this figure and the
# paper's Tier-2 one never say "recovered" in two different colours.
TRUE_BLUE, RECOVERED, MISSED, FALSE_POS, STRUCT, NODE = (
    "#0072B2", "#009E73", "#E69F00", "#CC79A7", "#9AA3AD", "#4A5A6A")


def _panel(fig, col, is_truth, seen):
    """One panel. `seen` is shared across both, so an outcome that only occurs in
    the right-hand panel still gets a legend entry."""

    def line(a, b, colour, dash, width, label, arc=False):
        xs, ys = (_arc(a, b, bow=0.5) if arc else ([a[0], b[0]], [a[1], b[1]]))
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode="lines", line=dict(color=colour, width=width, dash=dash),
            name=label, legendgroup=label, showlegend=label not in seen,
            hoverinfo="skip"), row=1, col=col)
        seen.add(label)

    for (p, c) in TREE_EDGES:
        if is_truth:
            line(PT[p], CT[c], TRUE_BLUE, "solid", 2.4, "true edge")
        elif (p, c) in tp:
            line(PT[p], CT[c], RECOVERED, "solid", 2.8, "recovered")
        else:
            line(PT[p], CT[c], MISSED, "dash", 2.4, "missed")

    if not is_truth:
        for (p, c) in sorted(fp):
            line(PT[p], CT[c], FALSE_POS, "dot", 2.4, "false positive")

    # (F) is structure inside the child block: no parent-block edge can express it,
    # and no gate in this composition scores it. Drawn so it is not silently absent.
    for (a, b), dash in ((( IN_BLOCK_PARENT, IN_BLOCK_CHILD), "solid"),
                         (IN_BLOCK_DUP, "dot")):
        line(CT[a], CT[b], STRUCT, dash, 1.4, "within-block structure (F)", arc=True)

    filled, hollow = [], []
    for idx, (x, y) in PT.items():
        (filled if is_truth or idx in {p for p, _ in survivors} else hollow).append(
            (x, y, f"P{idx}", idx))
    for idx, (x, y) in CT.items():
        (filled if is_truth or idx in {c for _, c in survivors} else hollow).append(
            (x, y, f"C{idx}", idx))

    for pts, face, edge, txt, label in (
            (filled, NODE, NODE, "white", "in the recovered graph"),
            (hollow, "white", STRUCT, STRUCT, "no surviving edge touches it")):
        if not pts:
            continue
        # The node key describes the RIGHT panel — on the left every feature is
        # filled by definition — so it is claimed by whichever panel first draws it
        # as an outcome, never by the ground-truth one.
        show = label not in seen and not is_truth
        fig.add_trace(go.Scatter(
            x=[q[0] for q in pts], y=[q[1] for q in pts], mode="markers+text",
            text=[str(q[3]) for q in pts], textposition="middle center",
            textfont=dict(color=txt, size=8),
            marker=dict(size=19, color=face, line=dict(width=1.4, color=edge)),
            name=label, legendgroup=label, showlegend=show,
            hovertext=[q[2] for q in pts], hoverinfo="text"), row=1, col=col)
        if show:
            seen.add(label)


fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.05,
    subplot_titles=(f"the tree behind the cache — {len(TREE_EDGES)} edges over "
                    f"{N_PARENT + N_CHILD} features",
                    f"after the battery — {len(tp)}/{len(TREE_EDGES)} recovered, "
                    f"{len(fn)} missed, {len(fp)} false"))
_seen = set()
_panel(fig, 1, True, _seen)
_panel(fig, 2, False, _seen)

# Both columns are indexed from 0 in their own block, so say which is which.
for col in (1, 2):
    for x, side in ((0.0, "parent block"), (1.0, "child block")):
        fig.add_annotation(x=x, y=0.85, text=side, showarrow=False, row=1, col=col,
                           font=dict(size=10, color=STRUCT))

fig.update_layout(
    width=1180, height=int(150 + 22 * N_ROWS), plot_bgcolor="white",
    margin=dict(l=20, r=20, t=70, b=100),
    legend=dict(orientation="h", yanchor="top", y=-0.01, x=0, font=dict(size=10)))
for c in (1, 2):
    fig.update_xaxes(visible=False, range=[-0.3, 1.45], row=1, col=c)
    fig.update_yaxes(visible=False, range=[-N_ROWS + 0.4, 1.25], row=1, col=c)
for note in fig.layout.annotations[:2]:
    note.font = dict(size=12, color=INK)
    note.xanchor, note.x = "left", note.x - 0.21

tree_fig = fig
tree_fig.show()

print(f"recovered   : {len(tp)}/{len(TREE_EDGES)}")
print(f"missed      : {sorted(fn)}   <- the absorbed edge; coverage never proposed it")
print(f"false       : {sorted(fp)}   <- the shared-topic pair no metric here rejects")
print(f"parent-block features with no true children: "
      f"{sorted(set(range(N_PARENT)) - set(TREE_CHILDREN))}")
print(f"child-block features with no true parent   : {ORPHANS}")

recovered   : 23/24
missed      : [(8, 21)]   <- the absorbed edge; coverage never proposed it
false       : [(9, 22)]   <- the shared-topic pair no metric here rejects
parent-block features with no true children: [6, 7, 9]
child-block features with no true parent   : [20, 22, 23, 27, 28, 29, 30, 31]


Both errors in the right panel are the two negative controls, and neither is a
threshold that could be tuned away:

- **8 → 21 comes out orange**, the only missed edge. Coverage cannot propose it —
  the absorbed child fires where its parent is silent — so no gate downstream ever
  saw it. §7 has the numbers.
- **9 → 22 comes out pink**, the only false positive, and every filter in the
  composition passes it because none of them tests conditional independence given
  a third cause.

Everything else is green, including 5 → 24, 25, 26: those edges are *correct*, and
the split is caught by metrics that flag the parent rather than by dropping its
edges — which is why the split parent is a filled node here and still shows up in
the scorecard at 98× on sibling redundancy.

### Which gate caught which disease

The figure above answers *did the battery return the tree?* — recovered, missed,
false. It cannot answer the question the world was built for: **which injected
structure ran into which gate?** Green tells you an edge survived; it does not tell
you that thirty grey pairs died at reconstruction and one amber pair died at the
frequency control.

So here is the same before/after in the **colours of §1**. The left panel is
literally the traces of that figure — one structure, one colour, all the way
through — and the right panel keeps every colour but changes the *style* to say what
happened to it:

- **solid, full strength** — the edge survived all three gates;
- **faded** — the gates removed it, and the note beside the parent says which one;
- **hollow node** — no surviving edge touches this feature.

Read it structure by structure and the battery's division of labour is visible:
each pathology is killed by exactly the gate it was designed to trip, and the two
negative controls are the two colours that come out the other side unchanged.

In [87]:
# The verdicts are read off the same masks §4 composed — nothing is restated by hand.
gate_recon_cut = edge_mask & ~m["recon"]["passes"]
gate_freq_cut = recon_ok & ~(survival >= C.FREQ_SURVIVAL_MIN)
true_children_of = {p: set(kids) for p, kids in TREE_CHILDREN.items()}


def parent_verdict(p):
    """One line per parent: what the gates did to its candidates, in order."""
    kept = int(freq_ok[p].sum())
    cut_recon = int(gate_recon_cut[p].sum())
    cut_freq = int(gate_freq_cut[p].sum())
    proposed = int(edge_mask[p].sum())
    parts = []
    if kept:
        parts.append(f"{kept} kept")
    if cut_recon:
        parts.append(f"{cut_recon} cut by reconstruction")
    if cut_freq:
        parts.append(f"{cut_freq} cut by frequency control")
    if not proposed and true_children_of.get(p):
        parts.append("never proposed by coverage")
    if p == SPLIT_PARENT:
        parts.append("parent flagged: redundancy")
    if p == TOPIC_PARENT and kept:
        parts.append("no gate tests this")
    return ", ".join(parts) or "no candidates"


def draw_world_vs_battery():
    """§1's figure, and the same world after the battery — same colours throughout."""
    declared = draw_world()                       # the very traces from §1
    fig = make_subplots(
        rows=1, cols=2, horizontal_spacing=0.06,
        subplot_titles=("declared — the six structures as injected",
                        f"after the battery — {len(tp)}/{len(TREE_EDGES)} true edges kept, "
                        f"{len(kept - survivors)} candidates cut"))

    for trace in declared.data:                   # left panel: nothing redrawn
        fig.add_trace(trace, row=1, col=1)

    # --- right panel: same colours, style carries the outcome -------------------
    def edge(pp, cc, role, dash, width, alive, ghost=0.40):
        """One edge. Removed ones keep their colour AND their dash -- the identity of
        the structure is the whole point -- so only opacity carries the outcome.

        `ghost` is per-role because ink accumulates: thirty overlapping superparent
        hairlines at the opacity that makes a single amber line readable would print
        as strong a wash as the left panel, and "the grey is gone" is exactly what
        this panel has to show.
        """
        colour = ROLE[role][0]
        fig.add_trace(go.Scatter(
            x=[PX, CX], y=[PY[pp], CY[cc]], mode="lines", showlegend=False,
            line=dict(color=colour, width=width if alive else max(width * 0.8, 1.0),
                      dash=dash),
            opacity=1.0 if alive else ghost,
            hovertext=f"{pp} -> {cc} — {'survived' if alive else 'removed'}",
            hoverinfo="text"), row=1, col=2)

    for c in range(N_CHILD):                                      # (A) superparent
        if bool(edge_mask[SUPERPARENT, c]):
            edge(SUPERPARENT, c, "superparent", "solid", 1.0,
                 (SUPERPARENT, c) in survivors, ghost=0.16)
    for (pp, cc) in GENUINE_EDGES:                                # the genuine tree
        edge(pp, cc, "genuine", "solid", 2.0, (pp, cc) in survivors)
    for cc in SPLIT_CHILDREN:                                     # (C) split
        edge(SPLIT_PARENT, cc, "split", "solid", 2.0,
             (SPLIT_PARENT, cc) in survivors)
    edge(FREQ_PARENT, FREQ_CHILD, "freq", "solid", 2.0,           # (B) coincidence
         (FREQ_PARENT, FREQ_CHILD) in survivors)
    edge(ABSORB_PARENT, ABSORB_CHILD, "absorbed", "dash", 2.0,    # (D) absorption
         (ABSORB_PARENT, ABSORB_CHILD) in survivors)
    edge(TOPIC_PARENT, TOPIC_CHILD, "topic", "dot", 2.0,          # (E) shared topic
         (TOPIC_PARENT, TOPIC_CHILD) in survivors)

    for (a, b), dash in (((IN_BLOCK_PARENT, IN_BLOCK_CHILD), "solid"),
                         (IN_BLOCK_DUP, "dot")):                  # (F) within-block
        xs, ys = _arc((CX, CY[a]), (CX, CY[b]))
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode="lines", showlegend=False, opacity=0.55,
            line=dict(color=ROLE["in_block"][0], width=1.6, dash=dash),
            hovertext=f"{a} — {b}: scored by the in-block metric, not by this "
                      f"composition", hoverinfo="text"), row=1, col=2)

    # Nodes keep their role colour -- the disease stays identifiable -- and go
    # hollow when no surviving edge touches them.
    live_p = {pp for pp, _ in survivors}
    live_c = {cc for _, cc in survivors}
    for xs, ys, roles, ids, live, side in (
            ([PX] * N_PARENT, [PY[i] for i in range(N_PARENT)],
             [parent_role(i) for i in range(N_PARENT)], list(range(N_PARENT)),
             live_p, "parent"),
            ([CX] * N_CHILD, [CY[i] for i in range(N_CHILD)],
             [child_role(i) for i in range(N_CHILD)], list(range(N_CHILD)),
             live_c, "child")):
        for alive in (True, False):
            keep = [j for j, i in enumerate(ids) if (i in live) == alive]
            if not keep:
                continue
            fig.add_trace(go.Scatter(
                x=[xs[j] for j in keep], y=[ys[j] for j in keep],
                mode="markers+text", showlegend=False,
                text=[str(ids[j]) for j in keep], textposition="middle center",
                textfont=dict(color="white" if alive else GREY, size=9),
                marker=dict(size=22,
                            color=[ROLE[roles[j]][0] if alive else "white" for j in keep],
                            line=dict(width=1.4,
                                      color=[ROLE[roles[j]][0] for j in keep])),
                hovertext=[f"{side}-local {ids[j]} — "
                           f"{'in the surviving graph' if alive else 'no surviving edge'}"
                           for j in keep], hoverinfo="text"), row=1, col=2)

    # The verdict beside each parent: which test its candidates went through.
    for pp in range(N_PARENT):
        fig.add_annotation(
            x=PX - 0.06, y=PY[pp], text=parent_verdict(pp), xanchor="right",
            showarrow=False, row=1, col=2,
            font=dict(size=8.5, color=ROLE[parent_role(pp)][0]))

    fig.update_layout(
        width=1420, height=int(150 + 24 * N_CHILD), plot_bgcolor="white",
        margin=dict(l=20, r=20, t=70, b=95),
        legend=dict(orientation="h", yanchor="top", y=-0.01, x=0, font=dict(size=10)))
    # One x-range for both panels: the verdict column needs room on the right, and
    # giving only that panel the room would offset every node between before and
    # after -- which is the one comparison this figure exists to make easy.
    for col in (1, 2):
        fig.update_xaxes(visible=False, range=[-1.25, 1.55], row=1, col=col)
    for col in (1, 2):
        fig.update_yaxes(visible=False, range=[-N_CHILD - 0.6, 1.2], row=1, col=col)
        for x, side in ((PX, "parent block"), (CX, "child block")):
            fig.add_annotation(x=x, y=0.75, text=f"<b>{side}</b>", showarrow=False,
                               row=1, col=col, font=dict(size=11, color=INK))
    for note in fig.layout.annotations[:2]:
        note.font = dict(size=12, color=INK)
        note.xanchor, note.x = "left", note.x - 0.24
    return fig


before_after_fig = draw_world_vs_battery()
before_after_fig.show()

print(f"{'structure':<26} {'designed to trip':<24} {'what actually acted'}")
print("-" * 92)
for label, designed, pp in (
        ("genuine tree (0-4)",      "nothing — must survive",   0),
        ("C feature split (5)",     "redundancy, energy",       SPLIT_PARENT),
        ("B freq. coincidence (6)", "frequency control",        FREQ_PARENT),
        ("A superparent (7)",       "reconstruction, out-deg",  SUPERPARENT),
        ("D absorption (8)",        "nothing — blind spot",     ABSORB_PARENT),
        ("E shared topic (9)",      "nothing — blind spot",     TOPIC_PARENT)):
    print(f"{label:<26} {designed:<24} {parent_verdict(pp)}")

structure                  designed to trip         what actually acted
--------------------------------------------------------------------------------------------
genuine tree (0-4)         nothing — must survive   4 kept
C feature split (5)        redundancy, energy       3 kept, parent flagged: redundancy
B freq. coincidence (6)    frequency control        1 cut by frequency control
A superparent (7)          reconstruction, out-deg  30 cut by reconstruction
D absorption (8)           nothing — blind spot     never proposed by coverage
E shared topic (9)         nothing — blind spot     1 kept, no gate tests this


Three readings the outcome-coloured figure could not give:

- **grey is gone.** Thirty superparent pairs entered the candidate set and none
  left it — all at gate 2, reconstruction, because a feature that fires on 91% of
  tokens with an activation of 0.015–0.03 contributes almost nothing to the child's
  reconstruction. Out-degree names it independently; the edges themselves die on
  reconstruction.
- **amber is gone, and only frequency could have removed it.** The
  coincidence edge passes coverage (R = 0.67) *and* reconstruction — its parent does
  contribute real decoder mass on those tokens. It dies at gate 3 with survival
  0.00, because outside the one high-frequency token id the pair never co-fires.
- **purple, red and blue are the honest part.** Purple survives and *should*:
  those are real edges, and the split is a property of the children, reported by a
  metric that flags the parent. Red never entered. Blue walked through every gate
  untouched. Two of the three are the negative controls, which is the same claim §7
  makes in numbers and this figure makes in colour.

## 5. The scorecard

`_score` grades fourteen rows. Each names a job, returns a verdict, and reports a
**margin**: how decisively that metric separated the class it should keep from the
class it should reject. A pass at 1.1× got the right answer this time; a pass at
>1000× would survive a very different world.

Three things about that column, because it is easy to over-read:

- **The verdict does not come from the margin.** Each row applies its own
  criterion and computes a margin from whatever two quantities it compared, so the
  numbers are not commensurable across rows. Metric 2b's margin is a ratio of
  *probe ranks*, and sits below 1 while the row passes — its criterion is that
  every genuine parent clears the top-*k* rule and the superparent clears it no
  more often than chance.
- **Two rows are scored categorically** — the recovered edge set is either right
  or it is not — and carry margin 1.0 meaning "correct", which on a ratio axis is
  indistinguishable from no separation at all. The dashboard plots them on a
  separate axis for that reason.
- **A margin of 1e9 is a floor, not a measurement**: it is what a row reports when
  the rejected class scored exactly zero and the division is against `1e-9`.

In [88]:
rows = _score(stats, labels, m)

w = max(len(r["metric"]) for r in rows)
print(f"{'metric':<{w}}  verdict  margin      job")
print("-" * (w + 60))
for r in sorted(rows, key=lambda r: (-int(r["pass"]), -r["margin"])):
    mg = ">1000x" if r["margin"] >= 1000 else f"{r['margin']:.1f}x"
    kind = " (cat.)" if r.get("margin_kind") == "categorical" else ""
    print(f"{r['metric']:<{w}}  {'PASS' if r['pass'] else 'FAIL':<7}  "
          f"{mg:<10}{kind}  {r['job']}")

n_pass = sum(r["pass"] for r in rows)
print(f"\n{n_pass}/{len(rows)} rows pass, covering all 21 metric functions.")

metric                                      verdict  margin      job
------------------------------------------------------------------------------------------------------
5. frequency control                        PASS     >1000x      reject frequency-coincidence edge, keep genuine
3'. parent-conditioned redundancy           PASS     >1000x      flag the split parent inside its own firing set, spare a genuine one
— absorption (negative control)             PASS     >1000x      confirm coverage cannot propose an absorbed edge at all
2. reconstruction                           PASS     >1000x      reject superparent edges, keep genuine
3. sibling redundancy                       PASS     98.0x       flag feature-split parent, spare healthy
6. independence null (PMI)                  PASS     41.0x       rank genuine edges above base-rate/superparent co-firing
9. energy concentration (share_energy)      PASS     3.7x        flag feature-split parent (a child holds >=90% of its energy)
8

The detail strings are where each row says what it actually measured, including
the caveats that do not fit in a verdict. Two are worth reading in full: metric 2b
explains why its own null rate is `k/D` and therefore not comparable across
dictionary sizes, and metric 7 explains that `r_supp` agreeing with
`joint_child_coverage_exact` is a drift guard rather than independent grading.

In [89]:
import textwrap

for r in rows:
    print(f"\n{r['metric']}  —  {'PASS' if r['pass'] else 'FAIL'}")
    print(textwrap.fill(r["detail"], 96, initial_indent="    ",
                        subsequent_indent="    "))


1. coverage (edge set)  —  PASS
    20/20 genuine edges kept; edge set also holds 35 non-genuine (that is what metrics 2-5 must
    prune)

2. reconstruction  —  PASS
    30/30 superparent edges rejected, 20/20 genuine kept (parent-gain: genuine>=3.35,
    superparent<=0.0027, thr=0.01)

3. sibling redundancy  —  PASS
    split parent redundancy=1.00 (flagged); healthy parents max=0.01 (thr=0.5)

4. out-degree / superparent  —  PASS
    detected superparents [7] (truth [7]); removing the superparent collapses Gini 0.562->0.356
    and top-1 share 55%->16% (covers degree_stats, gini)

5. frequency control  —  PASS
    1/1 freq edges rejected (survival=[0.0]); 20/20 genuine survive (min genuine survival=1.00,
    thr=0.5)

6. independence null (PMI)  —  PASS
    min genuine PMI=2.71 > max superparent PMI=0.07; base-rate confound only - topical co-
    occurrence is not in this toy (needs a model-based null)

7. joint-child coverage (support)  —  PASS
    genuine R_supp>=1.00 vs superpar

## 6. The published dashboard, inline

`reporting/visualize.py --calibration` builds the page that ships in
`outputs/synthetic_toy_calibration.html`: the full scorecard table, the margins on
a log axis, and one panel per metric that has two classes to separate. It calls
`build_world` and `_score` itself, so rendering it here re-runs the calibration
rather than reading a stale JSON — the figure below cannot disagree with the rows
printed above.

The strip panels are the part the table cannot show. A metric passes by putting
green above (or below) the dashed threshold and red on the other side; how much
daylight is between the two clouds is the margin, drawn instead of quoted.

In [90]:
from reporting.visualize import _calibration_data, build_calibration_dashboard

dash_data = _calibration_data()
dashboard_fig = build_calibration_dashboard(dash_data)
dashboard_fig.show()

## 7. The two negative controls

These rows pass when the battery does **not** do something. They exist because the
properties matrix claims two open columns, and a claim that nothing catches a
confound is worth exactly what a demonstration is worth.

**(D) Absorption.** The child took the parent's direction, so on the child's own
tokens the parent is silent — the operational signature of an absorbed feature.
`R = cofire / fire_c = 0`, so coverage never proposes the pair and metrics 2–9
never see it. This is not a threshold that could be loosened: fixing it needs a
different **candidate generator**, not a better grader. It is also why the label
set keeps `absorbed_edges` apart from `genuine` — scoring it as a miss would
penalise metrics for a candidate they were never handed.

**(E) Shared topic.** Both features are driven by a third thing and are
conditionally independent given it; neither refines the other. Reverse coverage is
1.0 because the child never fires off-topic, the parent contributes real decoder
mass, the pair spans buckets, and PMI is positive — so coverage, reconstruction,
frequency control and the independence null all pass it. Each tests a different
confound and none tests this one. Closing it needs a model-based topic null.

In [91]:
ap, ac = next(iter(labels.absorbed_edges))
tp, tc = next(iter(labels.topical_edges))

print(f"(D) absorption  {ap} -> {ac}   TRUE edge")
print(f"     cofire = {float(stats['cofire'][ap, ac]):.0f}, "
      f"fire(child) = {float(stats['fire_c'][ac]):.0f}  ->  "
      f"R = {float(R[ap, ac]):.2f} < tau = {C.EDGE_TAU}")
print(f"     proposed by coverage: {(ap, ac) in kept}   "
      f"(so nothing downstream is even asked)\n")

print(f"(E) shared topic {tp} -> {tc}   NOT an edge")
for gate, verdict in [
        ("coverage", (tp, tc) in kept),
        ("reconstruction", bool(recon_ok[tp, tc])),
        ("frequency control", bool(survival[tp, tc] >= C.FREQ_SURVIVAL_MIN)),
        ("independence null (PMI)", bool(m["pmi_valid"][tp, tc])
         and float(m["pmi"][tp, tc]) > 0)]:
    print(f"     {gate:<24} {'passes it' if verdict else 'rejects it'}")
print(f"     R = {float(R[tp, tc]):.2f}, PMI = {float(m['pmi'][tp, tc]):.2f}")

for r in rows:
    if r["metric"].lstrip().startswith("—"):
        print(f"\nscorecard: {r['metric']} -> {'PASS' if r['pass'] else 'FAIL'}")

(D) absorption  8 -> 21   TRUE edge
     cofire = 0, fire(child) = 25  ->  R = 0.00 < tau = 0.5
     proposed by coverage: False   (so nothing downstream is even asked)

(E) shared topic 9 -> 22   NOT an edge
     coverage                 passes it
     reconstruction           passes it
     frequency control        passes it
     independence null (PMI)  passes it
     R = 1.00, PMI = 3.69

scorecard: — absorption (negative control) -> PASS

scorecard: — topical co-occurrence (negative control) -> PASS


## 8. Does any of this depend on the seed?

`build_world(seed=...)` takes a seed, but nothing in the repo loops over one:
`calibrate()` is called once, at seed 0. The `validation/README.md` used to claim a
0–7 sweep that has never existed. This section is that sweep, and it belongs in a
notebook rather than in the script — the script is a gate that must stay fast, and
eight worlds cost eight times as much for a property that is not what it asserts.

The randomness is narrow by design: the tree, the pathologies and the token-id
plan are fixed, and the seed only moves activation magnitudes, the superparent's
firing mask, the decoder directions and the residual noise. So a verdict that
flipped here would mean a metric is passing on the noise draw — which is exactly
what the sweep is for. Watch the **margins**, not just the verdicts: a row whose
margin swings by orders of magnitude is a row whose threshold happens to sit in a
gap rather than a row that is measuring something stable.

In [92]:
SEEDS = list(range(8))
sweep = {}
for s in SEEDS:
    _, _, rs = calibrate(seed=s)
    sweep[s] = {r["metric"]: r for r in rs}
    fails = [r["metric"] for r in rs if not r["pass"]]
    print(f"seed {s}: {sum(r['pass'] for r in rs)}/{len(rs)} pass"
          + (f"   FAIL: {fails}" if fails else ""))

metrics_order = [r["metric"] for r in rows]
# A categorical row's margin is 1.0 meaning "correct", which shades like "no
# separation" on a ratio scale. Say so on the axis instead of in a caption.
row_label = {r["metric"]: r["metric"] + ("   (categorical)"
                                         if r.get("margin_kind") == "categorical" else "")
             for r in rows}
z = np.array([[min(sweep[s][name]["margin"], 1e4) for s in SEEDS]
              for name in metrics_order])
text = np.array([["PASS" if sweep[s][name]["pass"] else "FAIL" for s in SEEDS]
                 for name in metrics_order])

fig = go.Figure(go.Heatmap(
    z=np.log10(np.clip(z, 1e-3, None)), x=[f"seed {s}" for s in SEEDS],
    y=[row_label[name] for name in metrics_order], colorscale="Greens", zmin=0, zmax=4,
    colorbar=dict(title="log10 margin", thickness=14),
    text=text, texttemplate="%{text}", textfont=dict(size=9),
    hovertemplate="%{y}<br>%{x}<br>margin 10^%{z:.2f}<extra></extra>"))
fig.update_layout(
    title="Every row, every seed — verdict in the cell, margin in the shade",
    width=980, height=520, plot_bgcolor="white",
    yaxis=dict(autorange="reversed", tickfont=dict(size=10)),
    margin=dict(l=260, r=20, t=60, b=40))
sweep_fig = fig
sweep_fig.show()

print("\nmargin spread across seeds (ratio-scored rows):")
for name in metrics_order:
    ms = [sweep[s][name]["margin"] for s in SEEDS]
    if sweep[0][name].get("margin_kind") == "categorical":
        continue
    print(f"  {name:<38} min {min(ms):>12.2f}   max {max(ms):>12.2f}   "
          f"spread {max(ms) / max(min(ms), 1e-12):>8.1f}x")

seed 0: 14/14 pass
seed 1: 14/14 pass
seed 2: 14/14 pass
seed 3: 14/14 pass
seed 4: 14/14 pass
seed 5: 14/14 pass
seed 6: 14/14 pass
seed 7: 14/14 pass



margin spread across seeds (ratio-scored rows):
  2. reconstruction                      min       817.50   max      1489.97   spread      1.8x
  3. sibling redundancy                  min        98.03   max        98.03   spread      1.0x
  4. out-degree / superparent            min         1.58   max         1.58   spread      1.0x
  5. frequency control                   min 1000000000.00   max 1000000000.00   spread      1.0x
  6. independence null (PMI)             min        30.20   max        57.13   spread      1.9x
  7. joint-child coverage (support)      min         2.13   max         2.17   spread      1.0x
  8. joint-child coverage (energy)       min         2.12   max         2.17   spread      1.0x
  9. energy concentration (share_energy) min         3.55   max         3.77   spread      1.1x
  2b. probe S_res (rank rule)            min         0.67   max         4.00   spread      6.0x
  3'. parent-conditioned redundancy      min 1000000000.00   max 1000000000.00   spre

### Reading the sweep

Fourteen rows, eight worlds, no failures — the calibration is not living off seed
0. The margins say more than the verdicts, and they fall into three groups.

**Constant across every seed.** Sibling redundancy (98.0×), out-degree (1.6×), the
frequency control and parent-conditioned redundancy (both at the 1e9 floor). These
compare counts and set overlaps — a split whose children fire on identical
tokens, a superparent that reaches every child — and the genuine features' firing
pattern is laid out deterministically, so the seed cannot reach it. (It does move
the superparent's firing mask; that lands on ~90% of tokens either way, which is
why even the out-degree row comes out identical.) Little here *could* vary, which
is worth knowing before reading stability into it.

**Varying with the noise draw.** Reconstruction swings 817×–1490× and PMI 30×–57×,
because both read activation magnitudes and residual noise. Both stay two orders
of magnitude clear of their thresholds in every world.

**The one worth watching: metric 2b**, 0.67×–4.0×, the widest relative swing in the
sweep. It is a ratio of small integer ranks, so a single position's movement shows
up as a large factor — and the row passes in all eight worlds regardless, since its
criterion is the rank rule, not the ratio.

What the sweep cannot say anything about is the *design*: every world here has the
same tree, the same six pathologies and the same token plan. It rules out a lucky
noise draw, not a toy that is easier than the thing it stands in for. That is the
gap Tier 2 exists to narrow, by putting an SAE that had to *learn* the tree in
front of the same metrics.

## 9. Exporting the figures

Written into `validation/figures/tier1/`, which `.gitignore` covers (`figures/` is
ignored repo-wide) — these are rebuilt from the generator, not source. The training
notebook beside this one writes to `figures/tier2/`, so the two tiers' outputs never
land in one heap. PNG export needs `kaleido`.

**This notebook does not touch `outputs/`.** The published page and JSON are
written by the script's `main()`:

```bash
python3 validation/calibrate_on_synthetic_toy.py       # outputs/synthetic_toy_calibration.{md,json}
python3 -m reporting.visualize --calibration           # outputs/synthetic_toy_calibration.html
```

In [ ]:
# One directory per tier, so the two notebooks' outputs never sit in one heap:
# `figures/tier1/` here, `figures/tier2/` for the training notebook beside it.
FIG_DIR = ROOT / "validation" / "figures" / "tier1"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIGURES = {
    "toy_world_declared": world_fig,
    "toy_corpus_firing": corpus_fig,
    "toy_reverse_coverage": coverage_fig,
    "toy_gate_funnel": funnel_fig,
    "toy_tree_recovered": tree_fig,
    "toy_world_before_after": before_after_fig,
    "toy_calibration_dashboard": dashboard_fig,
    "toy_seed_sweep": sweep_fig,
}

for name, f in FIGURES.items():
    f.write_html(str(FIG_DIR / f"{name}.html"))
    try:
        f.write_image(str(FIG_DIR / f"{name}.png"), scale=3)
    except Exception as exc:                      # kaleido missing or headless
        print(f"  PNG skipped for {name}: {type(exc).__name__}: {exc}")

print("wrote", len(FIGURES), "figures to", FIG_DIR)

## What this notebook established, and what it did not

**Established.** Every metric function that ships runs against a world whose
answer was fixed before the metric saw it, at the thresholds production uses, and
each separates the class it should keep from the class it should reject —
fourteen rows across eight independent worlds. Coverage proposes all twenty
genuine edges and thirty-five others; reconstruction and the frequency control
remove exactly the pathological ones; the flagging metrics name the split parent
and the superparent by index.

**Not established.** That an SAE trained on real text would produce features with
this structure. Every pathology here was injected by hand, so Tier 1 structurally
cannot discover one nobody thought of — it can only confirm that the arithmetic
catches what it was built to catch. The two negative controls mark where the
battery is blind, and the seed sweep rules out luck, but neither makes the world
less artificial.

That is the whole reason the ladder has more rungs.
[`calibrate_on_trained_toy.py`](../calibrate_on_trained_toy.py) runs the same battery
on an SAE that had to learn its tree from activations, and found a defect nobody
injected: `parent_conditioned_redundancy` reports 0.958 for a parent whose two
true features co-fire zero times in the ground truth and 27,592 times in the
latents that recovered them. Tier 1 could never have produced that finding, and
Tier 2 could never have produced this one — that the metric reporting it is
sound.